# 🎮 Games by country, month by month — bot watch (GA4)

Pulls every **"pressed play"** game event from GA4 and shows how the countries playing the OLJ games change month by month.
The goal is to spot countries whose play numbers jump in a way that looks like bots. Every play is split **Web vs App**.

| Step | What it does |
|---|---|
| ⚙️ Settings | GA4 property, months to look at, **the event list** (check it!), flag thresholds |
| 1. GA4 connection | same login as the weekly brief (service account → OAuth fallback) |
| 2. Pull | 5 small GA4 queries: plays, players, game logins, all-site users — by month × country (× platform) |
| 2b. CMS | accounts created because of games: acquisition source contains a game name |
| 3. Event check | which configured events actually fire, on which platform, and anything that looks double-counted |
| 4. Table | one row per country × month with every measure |
| 5. Charts (optional) | interactive charts for exploring in the notebook — off by default, not in the report |
| 6. Suspicion table | each country's report month vs its own usual level, with flags |
| 7. Excel | all tables, for digging |
| 8. **Monthly Word report** | a new, self-contained `.docx` for the month, built around questions |

**Why these signals.** Bots almost always hit the website, not the native app. They rarely create an account or log in to a game,
and they tend to repeat the same action a lot. A real audience growing in a country should grow on Web *and* App,
with roughly the same plays per player and the same share of logged-in players. A bot surge usually breaks one or more of those.

**Each month:** run top to bottom on or after the 1st and you get `OLJ_Games_by_Country_YYYY-MM.docx` for the month that just ended.
To rebuild an older month, set `REPORT_MONTH` in ⚙️ Settings.

If GA4 can't be reached, the run **stops** — it never makes up numbers.

## Dependencies
Run once per Colab session.

In [ ]:
!pip install -q google-analytics-data google-auth google-auth-oauthlib plotly openpyxl python-docx matplotlib

## ⚙️ Settings

Everything you may want to change is in this one cell. **The event list is a first guess from the tracking plan.**
Check it, and delete or add lines as needed. Step 3 tells you which events GA4 actually has.

In [ ]:
import os
import datetime as dt

# ---- GA4 --------------------------------------------------------------------------------------------------
GA4_PROPERTY_ID = "328439412"      # GA4 Admin > Property Details (same as the weekly brief)
BRAND = "OLJ"                      # fixed: the games only exist on OLJ (used in titles and file names)

# ---- Period -----------------------------------------------------------------------------------------------
REPORT_MONTH = None                # None = automatic: the last complete month, based on today's date
                                   #   (run in Oct 2026 -> Sep 2025 → Sep 2026 · run in Nov 2026 -> Oct 2025 → Oct 2026)
                                   # "2026-08" = rebuild that month instead
MONTHS_BACK = 13                   # months in every table (13 = the same month last year is in the picture)

# ---- Notebook display -------------------------------------------------------------------------------------
SHOW_INTERACTIVE_CHARTS = False    # True = also draw the interactive charts in the notebook (step 5; never in the report)

# ---- Countries --------------------------------------------------------------------------------------------
TOP_N = 10                         # a country is shown if it's in the top N by plays in ANY month of the period,
                                   # so a country that suddenly jumps into the top N appears in the charts
MAX_COUNTRIES_SHOWN = 16           # hard cap on that list (ordered by total plays)

# ---- THE EVENTS: "user pressed play" ------------------------------------------------------------------------
# event name in GA4 : (game, where it's meant to fire, placement)
# "where" is what the tracking plan says. The actual Web/App split in every chart comes from GA4's `platform`
# dimension: web stream -> Web, iOS / Android (Firebase) streams -> App. A game page opened in the app's webview
# that fires the website's GA4 tag arrives through the web stream, so it counts as Web.
PLAY_EVENTS = {
    # Web -- "play" button on the game page (Tracking Plan (OLJ), Services > Jeux, rows 220-225)
    "jeux_motamo":            ("motamo",   "Web", "Game page"),
    "jeux_sudoku":            ("sudoku",   "Web", "Game page"),
    "jeux_prefixo":           ("prefixo",  "Web", "Game page"),
    "jeux_synonymo":          ("synonymo", "Web", "Game page"),
    "jeux_decrypto":          ("decrypto", "Web", "Game page"),
    "jeux_melimelo":          ("melimelo", "Web", "Game page"),
    # Web -- "play" buttons in the homepage games block (Tracking Plan (OLJ), rows 154-159)
    "jeux_motamo_hp":         ("motamo",   "Web", "Homepage block"),
    "jeux_sudoku_hp":         ("sudoku",   "Web", "Homepage block"),
    "jeux_prefixo_hp":        ("prefixo",  "Web", "Homepage block"),
    "jeux_synonymo_hp":       ("synonymo", "Web", "Homepage block"),
    "jeux_decrypto_hp":       ("decrypto", "Web", "Homepage block"),
    "jeux_melimelo_hp":       ("melimelo", "Web", "Homepage block"),
    # App -- game entry points on the Game List page (App plan, rows 184-189)
    "games_open_motamo":      ("motamo",   "App", "Game list"),
    "games_open_sudoku":      ("sudoku",   "App", "Game list"),
    "games_open_prefixo":     ("prefixo",  "App", "Game list"),
    "games_open_synonymo":    ("synonymo", "App", "Game list"),
    "games_open_decrypto":    ("decrypto", "App", "Game list"),
    "games_open_melimelo":    ("melimelo", "App", "Game list"),
    # App -- "Featured - Games Menu" on the app home (App plan, rows 35-40)
    "jeux_motamo_app_hp":     ("motamo",   "App", "App home menu"),
    "jeux_sudoku_app_hp":     ("sudoku",   "App", "App home menu"),
    "jeux_prefixo_app_hp":    ("prefixo",  "App", "App home menu"),
    "jeux_synonymo_app_hp":   ("synonymo", "App", "App home menu"),
    "jeux_decrypto_app_hp":   ("decrypto", "App", "App home menu"),
    "jeux_melimelo_app_hp":   ("melimelo", "App", "App home menu"),
}

# Game account / login events -- NOT plays. Used as a "real human" signal (bots rarely log in).
# (Tracking Plan (OLJ), games ias-container, rows 226-228)
GAME_ACCOUNT_EVENTS = [
    "olj_game_account_created",
    "olj_game_login",
    "olj_game_loggedin",
]

# Other game events in the tracking plan, deliberately NOT counted as plays (kept here for reference only):
#   jeux_view (opens a game page from the listing) · jeux_print · jeux_autres (other games thumbnail)
#   menu_jeux / submenu_nos_jeux / auj_nos_jeux / HP_banner_motamo / menu_motamo (navigation)

# ---- Suspicion flags (step 6 + report): report month vs the country's "usual" = median of the BASELINE_MONTHS before ----
BASELINE_MONTHS = 3
FLAG_MIN_PLAYS = 300           # ignore countries with fewer plays than this in the latest month
FLAG_PLAYS_GROWTH = 2.0        # plays at least 2x baseline
FLAG_APP_SHARE_DROP = 0.5      # app share of plays fell to half (or less) of baseline
FLAG_PPP_GROWTH = 1.5          # plays per player at least 1.5x baseline
FLAG_LOGGED_DROP = 0.5         # logged-in game users per player fell to half (or less) of baseline
FLAG_PLAYER_SITE_GROWTH = 2.0  # players as a share of the country's site users at least 2x baseline

# ---- CMS: accounts created because of games (report Q6) -- same API and key as the weekly brief ----------------
CMS_API_KEY = ""                   # leave EMPTY: read from the Colab 🔑 secret / GitHub secret CMS_API_KEY
CMS_API_HOST = "api.lorientlejour.com"
# The backend's /revenue/customer page URL, with the columns Q6 needs. {start}/{end} are filled in per chunk.
# Faster: if the backend can filter "acquisition source" (e.g. contains), apply it on that page and paste the URL.
CMS_ACCOUNTS_URL = (
    "https://managecmsnew.lorientlejour.com/revenue/customer?"
    "columns%5B%5D=id&columns%5B%5D=creationDate&columns%5B%5D=acquisitionSource&columns%5B%5D=source"
    "&columns%5B%5D=preferredLanguage"
    "&creationDate_operator=between&creationDate%5Bstart%5D={start}&creationDate%5Bend%5D={end}"
)
CMS_DATE_FMT = "%Y-%m-%d"
CMS_ACQ_FIELDS = ("acquisitionSource", "acquisition_source", "acquisition")   # the "acquisition source" column
CMS_CHUNK_DAYS = 3                 # days per CMS request (the CMS stops serving pages after ~1,000 rows)
GAME_ACCOUNT_MONTHS = MONTHS_BACK  # months in Q6 = same 13 months as the other tables (lower it if the CMS pull is too slow)
OT_LANGUAGES = {"en", "eng", "english", "anglais"}   # weekly brief rule: these preferred languages = OT accounts

# ---- Google service account (same as the weekly brief) --------------------------------------------------------
def _colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

_sa_key = (os.environ.get("GA4_SA_KEY") or _colab_secret("GA4_SA_KEY")).strip()
if _sa_key and not os.path.exists("service_account.json"):
    import json as _json
    try:
        _json.loads(_sa_key)
    except ValueError:
        raise RuntimeError("GA4_SA_KEY is set but isn't valid JSON -- paste the whole service-account key file into the secret")
    with open("service_account.json", "w") as _f:
        _f.write(_sa_key)
    print("GA4 key: written to service_account.json from the GA4_SA_KEY secret")

CMS_API_KEY = (os.environ.get("CMS_API_KEY") or _colab_secret("CMS_API_KEY") or CMS_API_KEY).strip()
if not CMS_API_KEY:
    raise RuntimeError("CMS -- no key: add the Colab 🔑 secret CMS_API_KEY (Notebook access on) / GitHub secret CMS_API_KEY")
if not (os.path.exists("service_account.json") or os.path.exists("oauth_client_secret.json")):
    raise RuntimeError("GA4 -- no key: add the Colab 🔑 secret GA4_SA_KEY (Notebook access on) "
                       "or upload service_account.json / oauth_client_secret.json with the 📁 Files panel")

# ---- Period, worked out from today ------------------------------------------------------------------------
_today = dt.date.today()
def _add_months(d, n):
    y, m = divmod(d.month - 1 + n, 12)
    return d.replace(year=d.year + y, month=m + 1, day=1)

if REPORT_MONTH:
    _report_start = dt.date(int(REPORT_MONTH[:4]), int(REPORT_MONTH[5:7]), 1)
else:
    _report_start = _add_months(_today.replace(day=1), -1)          # last complete month
if _report_start >= _today.replace(day=1):
    raise ValueError(f"REPORT_MONTH {REPORT_MONTH} isn't over yet -- pick a complete month or None")
# The pull always covers at least the trend months AND the months the "usual" level needs
_months_needed = max(MONTHS_BACK, BASELINE_MONTHS + 1)
PERIOD_START = _add_months(_report_start, -(_months_needed - 1))
PERIOD_END = _add_months(_report_start, 1) - dt.timedelta(days=1)   # last day of the report month
MONTHS = [_add_months(PERIOD_START, i).strftime("%Y%m") for i in range(_months_needed)]

print(f"Report month: {_report_start:%B %Y} | OLJ | data {PERIOD_START} → {PERIOD_END} ({_months_needed} months)")
print(f"{len(PLAY_EVENTS)} play events ({sum(v[1]=='Web' for v in PLAY_EVENTS.values())} web, "
      f"{sum(v[1]=='App' for v in PLAY_EVENTS.values())} app) + {len(GAME_ACCOUNT_EVENTS)} game-account events")

## 1. GA4 connection

Same login as the weekly brief, and its OLJ stream filter (stream name contains "olj"). The service account needs **Viewer** on the property; otherwise
`oauth_client_secret.json` logs in as you once.

In [ ]:
import pandas as pd
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    RunReportRequest, DateRange, Metric, Dimension, FilterExpression, FilterExpressionList, Filter,
)

SERVICE_ACCOUNT_FILE = "service_account.json"
OAUTH_CLIENT_SECRET_FILE = "oauth_client_secret.json"
OAUTH_TOKEN_FILE = "token.json"
GA4_SCOPES = ["https://www.googleapis.com/auth/analytics.readonly"]

def get_ga4_credentials():
    if os.path.exists(SERVICE_ACCOUNT_FILE):
        from google.oauth2 import service_account
        return service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=GA4_SCOPES)
    from google.auth.transport.requests import Request
    from google.oauth2.credentials import Credentials
    from google_auth_oauthlib.flow import InstalledAppFlow
    creds = Credentials.from_authorized_user_file(OAUTH_TOKEN_FILE, GA4_SCOPES) if os.path.exists(OAUTH_TOKEN_FILE) else None
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            creds = InstalledAppFlow.from_client_secrets_file(OAUTH_CLIENT_SECRET_FILE, GA4_SCOPES).run_local_server(port=0)
        with open(OAUTH_TOKEN_FILE, "w") as f:
            f.write(creds.to_json())
    return creds

ga4_client = BetaAnalyticsDataClient(credentials=get_ga4_credentials())

def _stream(match, value):
    mt = getattr(Filter.StringFilter.MatchType, match)
    return FilterExpression(filter=Filter(field_name="streamName",
                            string_filter=Filter.StringFilter(match_type=mt, value=value, case_sensitive=False)))

# OLJ streams only, same filter as the weekly brief: stream name contains "olj" (web + iOS + Android)
STREAM_FILTER = _stream("CONTAINS", "olj")

def event_filter(names):
    return FilterExpression(filter=Filter(field_name="eventName",
                            in_list_filter=Filter.InListFilter(values=list(names))))

def ga4_report(start, end, metrics, dimensions, extra_filter=None):
    """All rows (paged), metrics as numbers. Brand stream filter AND extra_filter."""
    parts = [f for f in (STREAM_FILTER, extra_filter) if f is not None]
    dim_filter = (None if not parts else parts[0] if len(parts) == 1
                  else FilterExpression(and_group=FilterExpressionList(expressions=parts)))
    rows, offset = [], 0
    while True:
        req = RunReportRequest(
            property=f"properties/{GA4_PROPERTY_ID}",
            date_ranges=[DateRange(start_date=str(start), end_date=str(end))],
            metrics=[Metric(name=m) for m in metrics],
            dimensions=[Dimension(name=d) for d in dimensions],
            limit=100000, offset=offset,
        )
        if dim_filter is not None:
            req.dimension_filter = dim_filter
        resp = ga4_client.run_report(req)
        for r in resp.rows:
            rows.append([v.value for v in r.dimension_values] + [v.value for v in r.metric_values])
        offset += len(resp.rows)
        if not resp.rows or offset >= resp.row_count:
            break
    df = pd.DataFrame(rows, columns=dimensions + metrics)
    for m in metrics:
        df[m] = pd.to_numeric(df[m], errors="coerce").fillna(0)
    return df

print(f"GA4 ready -- property {GA4_PROPERTY_ID}, OLJ streams (name contains 'olj')")

## 2. Pull from GA4

Eight queries over the whole period:

| Table | Dimensions | Metric | Filter |
|---|---|---|---|
| `ev` | month × country × platform × event | event count, users | play events |
| `players_cp` | month × country × platform | users | play events |
| `players_c` | month × country | users (de-duplicated across Web/App) | play events |
| `acct` | month × country | users | game-account events |
| `site` | month × country | users | none (all traffic of the brand) |
| `players_m` | month × platform, and month alone | users | play events (report Q2: players without country double counts) |
| `dev` | month × platform × device | event count | play events (report Q3: desktop / mobile / app) |
| `src` | month × platform × session source | event count | play events (report Q5: how they landed) |

A **player** = a GA4 user who fired at least one play event that month. A **play** = one play event.

In [ ]:
import pandas as pd
PLAY_NAMES = list(PLAY_EVENTS)

def surface(p):
    return "Web" if str(p).strip().lower() == "web" else "App"

ev = ga4_report(PERIOD_START, PERIOD_END, ["eventCount", "totalUsers"],
                ["yearMonth", "country", "platform", "eventName"], event_filter(PLAY_NAMES))
players_cp = ga4_report(PERIOD_START, PERIOD_END, ["totalUsers"], ["yearMonth", "country", "platform"],
                        event_filter(PLAY_NAMES))
players_c = ga4_report(PERIOD_START, PERIOD_END, ["totalUsers"], ["yearMonth", "country"], event_filter(PLAY_NAMES))
acct = ga4_report(PERIOD_START, PERIOD_END, ["totalUsers"], ["yearMonth", "country"], event_filter(GAME_ACCOUNT_EVENTS))
site = ga4_report(PERIOD_START, PERIOD_END, ["totalUsers"], ["yearMonth", "country"])
players_mp = ga4_report(PERIOD_START, PERIOD_END, ["totalUsers"], ["yearMonth", "platform"], event_filter(PLAY_NAMES))
players_m = ga4_report(PERIOD_START, PERIOD_END, ["totalUsers"], ["yearMonth"], event_filter(PLAY_NAMES))
dev = ga4_report(PERIOD_START, PERIOD_END, ["eventCount"], ["yearMonth", "platform", "deviceCategory"],
                 event_filter(PLAY_NAMES))
src = ga4_report(PERIOD_START, PERIOD_END, ["eventCount"], ["yearMonth", "platform", "sessionSource"],
                 event_filter(PLAY_NAMES))

if ev.empty:
    raise RuntimeError("GA4 returned NO play events at all for this brand and period -- check PLAY_EVENTS names "
                       "(GA4 > Reports > Engagement > Events) and the OLJ stream filter")

ev["surface"] = ev["platform"].map(surface)
ev["game"] = ev["eventName"].map(lambda e: PLAY_EVENTS[e][0])
ev["planned"] = ev["eventName"].map(lambda e: PLAY_EVENTS[e][1])
ev["placement"] = ev["eventName"].map(lambda e: PLAY_EVENTS[e][2])
players_cp["surface"] = players_cp["platform"].map(surface)

def month_label(ym):
    return dt.date(int(ym[:4]), int(ym[4:]), 1).strftime("%b %Y")

print(f"{len(ev):,} event rows | {ev['eventCount'].sum():,.0f} plays | {ev['country'].nunique()} countries | "
      f"months: {month_label(MONTHS[0])} → {month_label(MONTHS[-1])}")

## 2b. CMS — accounts created because of games

Pulls the accounts created in the last `GAME_ACCOUNT_MONTHS` months from `GET /cms/customer` (same API, key and
Cloudflare check as the weekly brief). An account counts for a game when its **acquisition source contains the game's
name**: motamo, sudoku, prefixo, synonymo, decrypto, melimelo. The match ignores case and accents. Months are in Beirut time.

It prints which field held the acquisition source and the values that matched, so you can check the rule.

In [ ]:
import requests, unicodedata
from IPython.display import display
from collections import Counter
from urllib.parse import urlsplit, parse_qsl
from zoneinfo import ZoneInfo

CMS_TZ = ZoneInfo("Asia/Beirut")
CMS_BASE_URL = f"https://{CMS_API_HOST}"
CMS_MAX_PAGES = 2000
GAMES = list(dict.fromkeys(v[0] for v in PLAY_EVENTS.values()))

def _cms_check(resp):
    if resp.status_code in (403, 503) and "just a moment" in resp.text[:2000].lower():
        raise RuntimeError("[CMS] Blocked by Cloudflare's bot check, not by the CMS -- the API key is fine. "
                           "Try Runtime -> Disconnect and delete runtime, then rerun.")
    resp.raise_for_status()

def _pick(c, *names):
    """First non-empty value for any of `names` (case-insensitive), top level then inside fields / preferences."""
    wanted = {n.lower() for n in names}
    for scope in (c, c.get("fields"), c.get("preferences")):
        if isinstance(scope, dict):
            for k, v in scope.items():
                if k.lower() in wanted and v not in (None, ""):
                    return v
    return None

def _created_local(c):
    raw = _pick(c, "creationDate", "creation_date", "created")
    if not raw:
        return None
    try:
        d = dt.datetime.fromisoformat(str(raw).replace("Z", "+00:00"))
    except ValueError:
        return None
    if d.tzinfo is None:
        d = d.replace(tzinfo=dt.timezone.utc)
    return d.astimezone(CMS_TZ).replace(tzinfo=None)

def _norm(x):
    return unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode().lower()

def acq_source(c):
    v = _pick(c, *CMS_ACQ_FIELDS)
    return "" if v is None else str(v)

def is_ot(c):
    """Weekly brief rule: English preferred language = OT account; anything else (blank, fr...) = OLJ."""
    lang = str(_pick(c, "preferredLanguage", "preferred_language", "language") or "").strip().lower()
    return lang in OT_LANGUAGES or lang.startswith(("en-", "en_"))

def game_of(c):
    a = _norm(acq_source(c))
    return next((g for g in GAMES if g in a), None)     # first game name found in the acquisition source

def _params(start, end):
    query = urlsplit(CMS_ACCOUNTS_URL).query
    fill = lambda v: v.replace("{start}", start.strftime(CMS_DATE_FMT)).replace("{end}", end.strftime(CMS_DATE_FMT))
    return [(k, fill(v)) for k, v in parse_qsl(query, keep_blank_values=True) if k != "page" and v != ""]

class TooManyRows(Exception):
    """The CMS stopped serving pages before every account in the date range came back (~1,000-row cap)."""

def cms_fetch(start, end, session, allow_partial=False):
    url = CMS_BASE_URL + "/cms/customer"
    out, seen, page, total = [], set(), None, 0
    for _ in range(CMS_MAX_PAGES):
        resp = session.get(url, params=_params(start, end) + ([("page", page)] if page is not None else []), timeout=60)
        _cms_check(resp)
        payload = resp.json()
        batch = []
        for c in payload.get("data") or []:
            key = str(_pick(c, "userId", "id") or id(c))
            if key not in seen:
                seen.add(key); batch.append(c)
        if not batch:
            break
        out += batch
        try:
            total = int(payload.get("total") or 0)
        except (TypeError, ValueError):
            total = 0
        if total and len(seen) >= total:
            break
        page = (payload["page"] if payload.get("page") is not None else (page or 0)) + 1
    if total and len(out) < total and not allow_partial:
        raise TooManyRows()
    return out, total

PARTIAL = []   # date ranges where the CMS cap cut the list short even at the smallest range: (start, end, got, total)

def cms_fetch_range(start, end, session):
    """Fetch [start, end]; when the CMS caps the answer, split the range in two and fetch each half,
    down to the smallest range (start day + 1). If even that is capped (a huge sign-up day), keep what came
    back and record it in PARTIAL -- the report flags those months as incomplete."""
    try:
        return cms_fetch(start, end, session)[0]
    except TooManyRows:
        if (end - start).days > 1:
            mid = start + dt.timedelta(days=(end - start).days // 2)
            return cms_fetch_range(start, mid, session) + cms_fetch_range(mid, end, session)   # halves overlap a day
        rows, total = cms_fetch(start, end, session, allow_partial=True)
        PARTIAL.append((start, end, len(rows), total))
        print(f"  ⚠️ {start:%Y-%m-%d} -> {end:%Y-%m-%d}: the CMS only returns {len(rows):,} of {total:,} accounts "
              f"(row cap) -- kept what came back")
        return rows

ACC_MONTHS = MONTHS[-GAME_ACCOUNT_MONTHS:]
_acc_start = dt.datetime.strptime(ACC_MONTHS[0] + "01", "%Y%m%d") - dt.timedelta(days=1)   # CMS filters in UTC
_acc_end = dt.datetime.combine(PERIOD_END, dt.time.min) + dt.timedelta(days=1)

session = requests.Session()
session.headers.update({"API-Key": CMS_API_KEY, "Accept": "application/json"})
customers, day, n_chunks = {}, _acc_start, 0
try:
    while day <= _acc_end:
        for c in cms_fetch_range(day, min(day + dt.timedelta(days=CMS_CHUNK_DAYS), _acc_end), session):
            customers.setdefault(str(_pick(c, "userId", "id") or id(c)), c)
        day += dt.timedelta(days=CMS_CHUNK_DAYS)
        n_chunks += 1
        if n_chunks % 10 == 0:
            print(f"  ... {day:%Y-%m-%d}: {len(customers):,} accounts so far")
except requests.HTTPError as e:
    raise RuntimeError(f"[CMS] failed (401/403 -> check CMS_API_KEY): {e}") from e

acc_rows = []
for c in customers.values():
    d = _created_local(c)
    if d is None:
        continue
    ym = d.strftime("%Y%m")
    if ym in ACC_MONTHS:
        acc_rows.append({"yearMonth": ym, "game": game_of(c), "acq": acq_source(c), "ot": is_ot(c)})
acc = pd.DataFrame(acc_rows, columns=["yearMonth", "game", "acq", "ot"])
if acc.empty:
    raise RuntimeError("[CMS] no accounts created in those months came back -- check CMS_ACCOUNTS_URL")

GAME_ACC = (acc.dropna(subset=["game"]).pivot_table(index="game", columns="yearMonth", values="acq", aggfunc="count")
              .reindex(index=GAMES, columns=ACC_MONTHS).fillna(0).astype(int))
# All new OLJ accounts: OLJ by language (weekly brief rule), plus any game account (games are OLJ-only)
ALL_ACC = acc[~acc["ot"] | acc["game"].notna()].groupby("yearMonth").size().reindex(ACC_MONTHS, fill_value=0)
# months touched by a capped range (their counts are too low) -- flagged in the report
PARTIAL_MONTHS = sorted({d.strftime("%Y%m") for s_, e_, _, _ in PARTIAL for d in (s_, e_)} & set(ACC_MONTHS))

_sample = next(iter(customers.values()))
_field = next((k for k in _sample if k.lower() in {f.lower() for f in CMS_ACQ_FIELDS}), None)
print(f"[CMS] {len(acc):,} accounts created {month_label(ACC_MONTHS[0])} → {month_label(ACC_MONTHS[-1])}, "
      f"{int(GAME_ACC.values.sum()):,} with a game in the acquisition source")
print("  acquisition-source field:", _field or f"NOT FOUND -- record fields are {sorted(_sample.keys())}; "
                                            "fix CMS_ACQ_FIELDS / add the column to CMS_ACCOUNTS_URL")
print("  values counted as games:", dict(Counter(acc.dropna(subset=["game"])["acq"]).most_common(15)))
print("  most common values overall:", dict(Counter(acc["acq"].replace("", "(blank)")).most_common(15)))
print(f"  OLJ accounts (all new accounts row): {int(ALL_ACC.sum()):,} · OT accounts left out: "
      f"{int((acc['ot'] & acc['game'].isna()).sum()):,} · game accounts with an English language: "
      f"{int((acc['ot'] & acc['game'].notna()).sum()):,} (kept)")
display(GAME_ACC.rename(columns=month_label))

## 3. Event check — are the configured events real, and where do they fire?

- **Never seen** = the name isn't in GA4 for this brand and period (typo, never shipped, or a different name in GA4).
- **Fires on the other platform** = e.g. a `jeux_*` event sent by the app itself (Firebase, iOS/Android), or a `games_open_*` event
  sent by the website. Then one tap may be counted twice (`games_open_*` + `jeux_*`). If you see a big share there, consider removing one of the two.
  (A game page opened in the app's webview that fires the website's tag counts as **Web**, so this check doesn't catch it.)

In [ ]:
check = (ev.groupby(["eventName", "surface"])["eventCount"].sum().unstack(fill_value=0)
           .reindex(columns=["Web", "App"], fill_value=0))
check = check.reindex(PLAY_NAMES, fill_value=0)
check.insert(0, "game", [PLAY_EVENTS[e][0] for e in check.index])
check.insert(1, "planned", [PLAY_EVENTS[e][1] for e in check.index])
check.insert(2, "placement", [PLAY_EVENTS[e][2] for e in check.index])
check["total"] = check["Web"] + check["App"]
check["% on other platform"] = [
    (row["App"] if row["planned"] == "Web" else row["Web"]) / row["total"] * 100 if row["total"] else 0
    for _, row in check.iterrows()]
by_month = ev.pivot_table(index="eventName", columns="yearMonth", values="eventCount", aggfunc="sum", fill_value=0)
check["months active"] = [(by_month.loc[e] > 0).sum() if e in by_month.index else 0 for e in check.index]

never = check.index[check["total"] == 0].tolist()
crossed = check.index[(check["total"] > 0) & (check["% on other platform"] >= 10)].tolist()
print("Never seen in GA4:", never or "none ✅")
print("Fires ≥10% on the other platform (possible double count):", crossed or "none ✅")

from IPython.display import display
display(check.style.format({"Web": "{:,.0f}", "App": "{:,.0f}", "total": "{:,.0f}", "% on other platform": "{:.0f}%"})
        .apply(lambda r: ["background:#fde2e1" if r["total"] == 0 else
                          ("background:#fff3cd" if r["% on other platform"] >= 10 else "") for _ in r], axis=1))

# Placement totals by month (shows if one entry point suddenly takes over)
place = ev.pivot_table(index="placement", columns="yearMonth", values="eventCount", aggfunc="sum", fill_value=0)
place.columns = [month_label(c) for c in place.columns]
display(place.style.format("{:,.0f}"))

# The game-account events
print("Game-account events found:", "yes" if acct["totalUsers"].sum() else "NONE -- the 'logged-in' chart will be empty")

## 4. Build the country × month table

One row per month × country, with these columns:

- **plays**, **plays_web**, **plays_app**
- **players**, **players_web**, **players_app**
- **app_share**: share of plays that came from the app
- **plays_per_player**
- **logged_per_100**: logged-in game users per 100 players
- **players_per_100_site**: players per 100 of that country's site users

In [ ]:
import numpy as np

KEY = ["yearMonth", "country"]
t = ev.pivot_table(index=KEY, columns="surface", values="eventCount", aggfunc="sum", fill_value=0)
t = t.reindex(columns=["Web", "App"], fill_value=0).rename(columns={"Web": "plays_web", "App": "plays_app"})
t["plays"] = t["plays_web"] + t["plays_app"]
pu = players_cp.pivot_table(index=KEY, columns="surface", values="totalUsers", aggfunc="sum", fill_value=0)
pu = pu.reindex(columns=["Web", "App"], fill_value=0).rename(columns={"Web": "players_web", "App": "players_app"})
t = (t.join(pu, how="left")
      .join(players_c.set_index(KEY)["totalUsers"].rename("players"), how="left")
      .join(acct.set_index(KEY)["totalUsers"].rename("logged_users"), how="left")
      .join(site.set_index(KEY)["totalUsers"].rename("site_users"), how="left")
      .fillna(0).reset_index())

def _div(a, b):
    return np.where(b > 0, a / np.where(b > 0, b, 1), np.nan)
t["app_share"] = _div(t["plays_app"], t["plays"]) * 100
t["plays_per_player"] = _div(t["plays"], t["players"])
t["logged_per_100"] = _div(t["logged_users"], t["players"]) * 100
t["players_per_100_site"] = _div(t["players"], t["site_users"]) * 100
t["share_of_plays"] = t["plays"] / t.groupby("yearMonth")["plays"].transform("sum") * 100
t["rank"] = t.groupby("yearMonth")["plays"].rank(ascending=False, method="first").astype(int)

# Countries to show: in the top N of ANY month, ordered by total plays
ever_top = t.loc[t["rank"] <= TOP_N, "country"].unique()
COUNTRIES = (t[t["country"].isin(ever_top)].groupby("country")["plays"].sum()
               .sort_values(ascending=False).index[:MAX_COUNTRIES_SHOWN].tolist())
MONTH_LABELS = [month_label(m) for m in MONTHS]

# All-countries totals, for reference lines
tot = t.groupby("yearMonth")[["plays", "plays_app", "players", "logged_users"]].sum()
tot["app_share"] = tot["plays_app"] / tot["plays"] * 100

def grid(col, countries=None):
    g = t.pivot_table(index="country", columns="yearMonth", values=col, aggfunc="sum")
    g = g.reindex(index=countries or COUNTRIES, columns=MONTHS)
    g.columns = MONTH_LABELS
    return g

print(f"{len(COUNTRIES)} countries shown (top {TOP_N} in at least one month):", ", ".join(COUNTRIES))
newcomers = [c for c in COUNTRIES if t[(t["country"] == c) & (t["rank"] <= TOP_N)]["yearMonth"].min() != MONTHS[0]]
print("Entered the top", TOP_N, "after the first month:", newcomers or "none")

## 5. Interactive charts (optional — only if `SHOW_INTERACTIVE_CHARTS = True`)

For exploring in the notebook; they don't go in the report. Read them in this order:

1. **Share of plays** heatmap: who plays, and how it shifts.
2. **Rank** lines: newcomers jumping into the top N.
3. **Web vs App per country**: a bot surge is almost always a web-only bar shooting up.
4. **App share**, **plays per player**, and **logged-in players**: the "does it look human" checks.
5. **Game mix**: bots often hammer a single game.

In [ ]:
if not SHOW_INTERACTIVE_CHARTS:
    print("Interactive charts skipped (SHOW_INTERACTIVE_CHARTS = False in ⚙️ Settings). The report has its own chart.")
else:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    FIGS = {}
    def heat(g, title, fmt, colorscale="Blues", zmid=None, hover_extra=None, unit=""):
        text = g.map(lambda v: "" if pd.isna(v) else format(v, fmt))
        fig = go.Figure(go.Heatmap(
            z=g.values, x=g.columns, y=g.index, text=text.values, texttemplate="%{text}",
            colorscale=colorscale, zmid=zmid, hoverongaps=False,
            customdata=hover_extra.values if hover_extra is not None else None,
            hovertemplate="%{y} · %{x}<br>%{text}" + unit + ("<br>plays: %{customdata:,.0f}" if hover_extra is not None else "")
                          + "<extra></extra>"))
        fig.update_layout(title=title, height=110 + 30 * len(g), yaxis_autorange="reversed",
                          margin=dict(l=10, r=10, t=60, b=10), xaxis_side="top")
        return fig

    # 1. Share of plays
    FIGS["share"] = heat(grid("share_of_plays"), f"1 · Share of all plays (%) — top {TOP_N} countries of any month",
                         ".1f", "Blues", hover_extra=grid("plays"), unit="%")
    FIGS["share"].show()

    # 2. Rank over time (only ranks ≤ TOP_N drawn, so entering / leaving the top N is visible)
    r = grid("rank")
    fig = go.Figure()
    for c in COUNTRIES:
        y = r.loc[c].where(r.loc[c] <= TOP_N)
        fig.add_trace(go.Scatter(x=MONTH_LABELS, y=y, mode="lines+markers", name=c, connectgaps=False,
                                 hovertemplate=f"{c}<br>%{{x}}: #%{{y}}<extra></extra>"))
    fig.update_layout(title=f"2 · Rank by plays (top {TOP_N} only)", yaxis=dict(autorange="reversed", dtick=1, title="rank"),
                      height=480, margin=dict(l=10, r=10, t=60, b=10))
    FIGS["rank"] = fig; fig.show()

    # 3. Web vs App plays per country (small multiples, each country on its own scale)
    cols = 4
    rows = int(np.ceil(len(COUNTRIES) / cols))
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=COUNTRIES, shared_xaxes=True,
                        vertical_spacing=0.08 if rows > 1 else 0.2, horizontal_spacing=0.05)
    web, app = grid("plays_web"), grid("plays_app")
    for i, c in enumerate(COUNTRIES):
        rr, cc = i // cols + 1, i % cols + 1
        for name, g, color in (("Web", web, "#4C78A8"), ("App", app, "#F58518")):
            fig.add_trace(go.Bar(x=MONTH_LABELS, y=g.loc[c].fillna(0), name=name, marker_color=color,
                                 showlegend=(i == 0), legendgroup=name,
                                 hovertemplate=f"{c} · %{{x}}<br>{name}: %{{y:,.0f}}<extra></extra>"), row=rr, col=cc)
    fig.update_layout(barmode="stack", title="3 · Plays per country — Web vs App (each panel has its own scale)",
                      height=230 * rows + 80, margin=dict(l=10, r=10, t=80, b=10))
    fig.update_xaxes(showticklabels=False)
    FIGS["web_app"] = fig; fig.show()

    # 4. App share of plays
    avg = tot["app_share"].mean()
    FIGS["app_share"] = heat(grid("app_share"), f"4 · % of plays from the App (all countries ≈ {avg:.0f}%) — red = mostly web",
                             ".0f", "RdBu", zmid=avg, hover_extra=grid("plays"), unit="%")
    FIGS["app_share"].show()

    # 5. Plays per player
    FIGS["ppp"] = heat(grid("plays_per_player"), "5 · Plays per player — a jump = the same 'users' replaying a lot",
                       ".1f", "OrRd", hover_extra=grid("plays"))
    FIGS["ppp"].show()

    # 6. Logged-in game users per 100 players
    FIGS["logged"] = heat(grid("logged_per_100"), "6 · Logged-in game users per 100 players — a drop = more anonymous players",
                          ".0f", "Greens", hover_extra=grid("plays"))
    FIGS["logged"].show()

    # 7. Players as a share of the country's site users
    FIGS["site"] = heat(grid("players_per_100_site"), "7 · Players per 100 site users of that country — a jump = games-only traffic",
                        ".1f", "Purples", hover_extra=grid("plays"))
    FIGS["site"].show()

    # 8. Game mix in the latest month: each shown country vs all countries
    last = MONTHS[-1]
    gm = ev[ev["yearMonth"] == last].pivot_table(index="country", columns="game", values="eventCount", aggfunc="sum", fill_value=0)
    gm.loc["ALL COUNTRIES"] = gm.sum()
    gm = gm.reindex(["ALL COUNTRIES"] + COUNTRIES).fillna(0)
    gm = gm.div(gm.sum(axis=1).replace(0, np.nan), axis=0) * 100
    long = gm.reset_index().melt(id_vars="country", var_name="game", value_name="share")
    fig = px.bar(long, y="country", x="share", color="game", orientation="h",
                 title=f"8 · Game mix in {month_label(last)} (% of each country's plays)")
    fig.update_layout(height=120 + 28 * len(gm), yaxis=dict(autorange="reversed", title=""), xaxis_title="%",
                      margin=dict(l=10, r=10, t=60, b=10))
    FIGS["games"] = fig; fig.show()

## 6. Suspicion table — latest month vs each country's own baseline

The baseline is the median of the previous `BASELINE_MONTHS` months. A flag is raised when a country (with at least `FLAG_MIN_PLAYS` plays):

| Flag | Meaning |
|---|---|
| 📈 surge | plays ≥ `FLAG_PLAYS_GROWTH` × baseline |
| 🌐 web-only | app share fell to ≤ `FLAG_APP_SHARE_DROP` × baseline |
| 🔁 replays | plays per player ≥ `FLAG_PPP_GROWTH` × baseline |
| 👤 anonymous | logged-in per player fell to ≤ `FLAG_LOGGED_DROP` × baseline |
| 🎯 games-only | players per site user ≥ `FLAG_PLAYER_SITE_GROWTH` × baseline |
| 🆕 new | no baseline at all (country wasn't playing before) |

**One flag** = worth a look. **Two or more together** (typically 📈 + 🌐, or 📈 + 🔁 + 👤) is the classic bot pattern.
To review an earlier month, set `REPORT_MONTH` in ⚙️ Settings.

In [ ]:
LATEST = MONTHS[-1]          # the report month (set REPORT_MONTH in ⚙️ Settings to change it)
li = MONTHS.index(LATEST)
base_months = MONTHS[max(0, li - BASELINE_MONTHS):li]
COLS = ["plays", "app_share", "plays_per_player", "logged_per_100", "players_per_100_site"]

cur = t[t["yearMonth"] == LATEST].set_index("country")[["rank"] + COLS]
base = t[t["yearMonth"].isin(base_months)].groupby("country")[COLS].median()
s = cur.join(base, rsuffix="_base", how="left")
s = s[s["plays"] >= FLAG_MIN_PLAYS].sort_values("plays", ascending=False)

def ratio(a, b):
    return a / b if pd.notna(a) and pd.notna(b) and b > 0 else np.nan

flags = []
for c, r in s.iterrows():
    f = []
    if pd.isna(r["plays_base"]) or r["plays_base"] == 0:
        f.append("🆕 new")
    else:
        if ratio(r["plays"], r["plays_base"]) >= FLAG_PLAYS_GROWTH: f.append("📈 surge")
        x = ratio(r["app_share"], r["app_share_base"])
        if pd.notna(x) and x <= FLAG_APP_SHARE_DROP: f.append("🌐 web-only")
        x = ratio(r["plays_per_player"], r["plays_per_player_base"])
        if pd.notna(x) and x >= FLAG_PPP_GROWTH: f.append("🔁 replays")
        x = ratio(r["logged_per_100"], r["logged_per_100_base"])
        if pd.notna(x) and x <= FLAG_LOGGED_DROP: f.append("👤 anonymous")
        x = ratio(r["players_per_100_site"], r["players_per_100_site_base"])
        if pd.notna(x) and x >= FLAG_PLAYER_SITE_GROWTH: f.append("🎯 games-only")
    flags.append(" ".join(f))
s["flags"] = flags
s["n_flags"] = s["flags"].apply(lambda x: sum(x.count(e) for e in ("🆕", "📈", "🌐", "🔁", "👤", "🎯")))
s["plays_vs_base"] = [ratio(a, b) for a, b in zip(s["plays"], s["plays_base"])]

SUSPICION = s[["rank", "plays", "plays_base", "plays_vs_base", "app_share", "app_share_base",
               "plays_per_player", "plays_per_player_base", "logged_per_100", "logged_per_100_base",
               "players_per_100_site", "players_per_100_site_base", "flags", "n_flags"]]

print(f"{month_label(LATEST)} vs median of {', '.join(month_label(m) for m in base_months) or 'nothing (first month)'}")
print("Countries with 2+ flags:", SUSPICION.index[SUSPICION["n_flags"] >= 2].tolist() or "none")
display(SUSPICION.style
        .format({"plays": "{:,.0f}", "plays_base": "{:,.0f}", "plays_vs_base": "{:.1f}×",
                 "app_share": "{:.0f}%", "app_share_base": "{:.0f}%",
                 "plays_per_player": "{:.1f}", "plays_per_player_base": "{:.1f}",
                 "logged_per_100": "{:.0f}", "logged_per_100_base": "{:.0f}",
                 "players_per_100_site": "{:.1f}", "players_per_100_site_base": "{:.1f}"}, na_rep="–")
        .apply(lambda r: ["background:#fde2e1" if r["n_flags"] >= 2 else
                          ("background:#fff3cd" if r["n_flags"] == 1 else "") for _ in r], axis=1))

## 7. Excel — every table, for digging (optional)

In [ ]:
xlsx = f"{BRAND}_Games_by_Country_{MONTHS[-1][:4]}-{MONTHS[-1][4:]}_data.xlsx"
with pd.ExcelWriter(xlsx) as w:
    SUSPICION.to_excel(w, sheet_name="Suspicion " + LATEST)
    for col, name in (("plays", "Plays"), ("plays_web", "Plays Web"), ("plays_app", "Plays App"),
                      ("share_of_plays", "Share %"), ("rank", "Rank"), ("players", "Players"),
                      ("app_share", "App share %"), ("plays_per_player", "Plays per player"),
                      ("logged_per_100", "Logged per 100"), ("players_per_100_site", "Players per 100 site")):
        grid(col).to_excel(w, sheet_name=name)
    t.to_excel(w, sheet_name="All countries (long)", index=False)
    check.to_excel(w, sheet_name="Event check")
    ev.to_excel(w, sheet_name="Raw events", index=False)
print("Saved:", xlsx)

## 8. Monthly Word report

Builds a **new** `.docx` for the report month: `OLJ_Games_by_Country_YYYY-MM.docx`. Each month gets its own file.

- **Layout.** Same font and colours as the Stories of Love report. Every table is titled by its question.
- **Notes.** Explanations sit in the **Notes** appendix, referenced by `*1`, `*2`, … in the tables.
- **Page breaks.** Header rows repeat on every page, and rows never split.

The questions are being added one at a time. So far the report has:

1. **How many games were played each month?** · Web / App / Total, one row per month
2. **How many players were there each month?** · same layout
3. **Where did they play?** · Desktop / Mobile / App, one row per month
4. **Where are they?** · each month's own top 5 (★ new, ↩ back, who left), other
5. **How did they land here?** · traffic-source groups (same lists as the weekly brief), one colourful table
6. **How many accounts were created because of games?** · CMS, by game + all games + all accounts, one colourful table

Every question uses the same layout: months are the rows, and each column group shows the number and its change vs the
previous month. When a question has more than 3 groups, it is split into several tables of 3, so the font never shrinks.

In [ ]:
import numpy as np
from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# ---------------------------------------------------------------------------------------------------------
# Look: Stories of Love font + sizes, weekly brief colours for Web / App / Total
# ---------------------------------------------------------------------------------------------------------
FONT = "Calibri"                      # the Stories of Love report's font (Word's theme body font)
NAVY, GREY, DARK_GREY, LINE = "2B3440", "8A9096", "5F6368", "E3E6EA"
UP, DOWN = "137333", "C5221F"
PAL = {   # group header · sub-header fill · sub-header text · cell fill   (weekly brief)
    "web":   ("1A73E8", "D6E4FB", "1A5BB8", "F0F5FE"),
    "app":   ("1E8E3E", "D3ECD9", "17702F", "F0F8F2"),
    "total": ("2B3440", "DDE1E6", "2B3440", "F3F4F6"),
}
PAL["desktop"] = PAL["web"]
PAL["mobile"] = ("7E57C2", "E4DAF6", "5E35B1", "F6F2FF")
PAGE_W = 21.0 - 2 * 1.5               # A4 minus Stories of Love margins (cm)

LM = MONTHS[-1]
LM_LONG = dt.date(int(LM[:4]), int(LM[4:]), 1).strftime("%B %Y")

# ---------------------------------------------------------------------------------------------------------
# Low-level helpers
# ---------------------------------------------------------------------------------------------------------
doc = Document()
sec = doc.sections[0]
sec.page_width, sec.page_height = Cm(21), Cm(29.7)
sec.left_margin = sec.right_margin = Cm(1.5)
sec.top_margin = sec.bottom_margin = Cm(1.0)
_normal = doc.styles["Normal"]
_normal.font.name, _normal.font.size = FONT, Pt(9)
_normal.font.color.rgb = RGBColor.from_string(NAVY)
_normal.element.rPr.rFonts.set(qn("w:eastAsia"), FONT)
_normal.paragraph_format.space_after = Pt(2)

def run(p, text, size=9, bold=False, color=NAVY, sup=False):
    r = p.add_run(text)
    r.font.name, r.font.size, r.font.bold = FONT, Pt(size), bold
    r.font.color.rgb = RGBColor.from_string(color)
    r.font.superscript = sup
    return r

def para(text="", size=9, bold=False, color=NAVY, before=0, after=2, keep_next=False):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.space_before, pf.space_after, pf.keep_with_next = Pt(before), Pt(after), keep_next
    if text:
        run(p, text, size, bold, color)
    return p

def question(text):
    """Every table starts with its question -- nothing else above it."""
    para(text, 11, True, NAVY, before=14, after=4, keep_next=True)

def _shade(cell, fill):
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto"); shd.set(qn("w:fill"), fill)
    cell._tc.get_or_add_tcPr().append(shd)

def _border(cell, edge, color, size):
    tcPr = cell._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders"); tcPr.append(b)
    e = OxmlElement(f"w:{edge}")
    e.set(qn("w:val"), "single"); e.set(qn("w:sz"), str(size)); e.set(qn("w:space"), "0"); e.set(qn("w:color"), color)
    b.append(e)

def _row_props(row, header=False):
    trPr = row._tr.get_or_add_trPr()
    trPr.append(OxmlElement("w:cantSplit"))          # a row never breaks across pages
    if header:
        trPr.append(OxmlElement("w:tblHeader"))      # header rows repeat on every page

def _table_borders(tb):
    b = OxmlElement("w:tblBorders")
    for edge, val in (("top", "nil"), ("left", "nil"), ("right", "nil"), ("insideV", "nil"),
                      ("bottom", "single"), ("insideH", "single")):
        e = OxmlElement(f"w:{edge}"); e.set(qn("w:val"), val)
        if val == "single":
            e.set(qn("w:sz"), "4"); e.set(qn("w:space"), "0"); e.set(qn("w:color"), LINE)
        b.append(e)
    tb._tbl.tblPr.append(b)
    lay = OxmlElement("w:tblLayout"); lay.set(qn("w:type"), "fixed"); tb._tbl.tblPr.append(lay)

def _write(cell, parts, align=WD_ALIGN_PARAGRAPH.CENTER):
    """parts = list of (text, size, bold, color, superscript)"""
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
    p = cell.paragraphs[0]
    p.alignment = align
    p.paragraph_format.space_after = Pt(0)
    for text, size, bold, color, sup in parts:
        run(p, text, size, bold, color, sup)

def label(text, note=None, size=8, bold=True, color=NAVY):
    """Cell content: text + optional *n note marker (superscript, grey)."""
    parts = [(text, size, bold, color, False)]
    if note:
        parts.append((f"*{note}", size - 1, False, GREY, True))
    return parts

# ---------------------------------------------------------------------------------------------------------
# Notes (*1, *2 ...) -- collected while building, printed in the appendix
# ---------------------------------------------------------------------------------------------------------
NOTES = {}
def note(key, text):
    """Register an explanation once; returns its number for the *n marker."""
    if key not in NOTES:
        NOTES[key] = (len(NOTES) + 1, text)
    return NOTES[key][0]

# ---------------------------------------------------------------------------------------------------------
# Change cell: ▲ green / ▼ red, as in the weekly brief
# ---------------------------------------------------------------------------------------------------------
def change_parts(cur, prev, size=8, compact=False):
    if prev is None or (isinstance(prev, float) and np.isnan(prev)):
        return [("–", size, False, GREY, False)]
    if prev == 0:
        return [("new", size, True, UP, False)] if cur > 0 else [("–", size, False, GREY, False)]
    c = round((cur / prev - 1) * 100)
    if c == 0:
        return [("0%", size, True, DARK_GREY, False)]
    if compact and c >= 1000:                        # narrow columns: +10,491% is written ×106 (106 times as many)
        return [(f"▲ ×{cur / prev:,.0f}", size, True, UP, False)]
    return [(f"{'▲' if c > 0 else '▼'} {abs(c):,}%", size, True, UP if c > 0 else DOWN, False)]

# ---------------------------------------------------------------------------------------------------------
# Grouped table: one label column + groups (Web / App / Total), weekly-brief style. Reused by every question.
# ---------------------------------------------------------------------------------------------------------
def grouped_table(first_header, groups, rows, first_w=3.0, head_size=7.5, sub_size=7, cell_mar=None, sub_widths=None):
    """first_header: parts for the top-left cell
    groups: [(palette key, header parts, [sub-header parts, ...]), ...]
    rows:   [(label parts, [cell parts for every sub-column, in order]), ...]"""
    n_sub = [len(g[2]) for g in groups]
    ncols = 1 + sum(n_sub)
    sub_w = (PAGE_W - first_w) / sum(n_sub)
    tb = doc.add_table(rows=2 + len(rows), cols=ncols)
    tb.alignment = WD_TABLE_ALIGNMENT.CENTER
    tb.autofit = False
    _table_borders(tb)
    if cell_mar is not None:                         # narrower left/right cell padding (cm), for wide tables
        mar = OxmlElement("w:tblCellMar")
        for edge in ("left", "right"):
            e = OxmlElement(f"w:{edge}"); e.set(qn("w:w"), str(int(cell_mar * 567))); e.set(qn("w:type"), "dxa")
            mar.append(e)
        tb._tbl.tblPr.append(mar)
    _row_props(tb.rows[0], header=True)
    _row_props(tb.rows[1], header=True)

    # header: top-left cell spans both header rows
    tl = tb.cell(0, 0).merge(tb.cell(1, 0))
    _write(tl, first_header, WD_ALIGN_PARAGRAPH.LEFT)
    _border(tl, "bottom", NAVY, 10)
    col = 1
    for (key, head, subs), n in zip(groups, n_sub):
        g_head, s_fill, s_text, _ = PAL[key] if isinstance(key, str) else key
        top = tb.cell(0, col).merge(tb.cell(0, col + n - 1)) if n > 1 else tb.cell(0, col)
        _write(top, [(t, head_size, True, "FFFFFF", sup) for t, _, _, _, sup in head])
        _shade(top, g_head)
        _border(top, "left", "FFFFFF", 18)
        for k, sub in enumerate(subs):
            c = tb.cell(1, col + k)
            _write(c, [(t, sub_size, True, s_text, sup) for t, _, _, _, sup in sub])
            _shade(c, s_fill)
            if k == 0:
                _border(c, "left", "FFFFFF", 18)
        col += n

    # body
    for i, (lab, cells) in enumerate(rows):
        r = tb.rows[2 + i]
        _row_props(r)
        _write(r.cells[0], lab, WD_ALIGN_PARAGRAPH.LEFT)
        col = 1
        for (key, _, subs), n in zip(groups, n_sub):
            fill = (PAL[key] if isinstance(key, str) else key)[3]
            for k in range(n):
                c = r.cells[col + k]
                spec = cells[col - 1 + k]
                if isinstance(spec, dict):               # {"parts": [...], "fill": "RRGGBB"} = own colour
                    _write(c, spec["parts"])
                    _shade(c, spec["fill"])
                else:
                    _write(c, spec)
                    _shade(c, fill)
                if k == 0:
                    _border(c, "left", "FFFFFF", 18)
            col += n

    # keep each table on one page when it fits (it still breaks, with repeated headers, if it doesn't)
    for row in tb.rows[:-1]:
        for cell in row.cells:
            for p in cell.paragraphs:
                p.paragraph_format.keep_with_next = True

    sws = sub_widths or [sub_w] * sum(n_sub)
    widths = [first_w] + list(sws)
    for j, w in enumerate(widths):
        tb.columns[j].width = Cm(w)                  # grid
    for row in tb.rows[1:]:
        for j, w in enumerate(widths):
            if j or row is not tb.rows[1]:
                row.cells[j].width = Cm(w)
    tl.width = Cm(first_w)
    col = 1
    for n in n_sub:                                  # merged group headers = sum of their columns
        tb.cell(0, col).width = Cm(sum(sws[col - 1:col - 1 + n]))
        col += n
    return tb

def footer():
    p = sec.footer.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run(p, f"{BRAND} games · {LM_LONG} · page ", 7, color=GREY)
    r = run(p, "", 7, color=GREY)
    for kind, val in (("begin", None), ("instr", "PAGE"), ("end", None)):
        if kind == "instr":
            it = OxmlElement("w:instrText"); it.set(qn("xml:space"), "preserve"); it.text = val; r._r.append(it)
        else:
            fc = OxmlElement("w:fldChar"); fc.set(qn("w:fldCharType"), kind); r._r.append(fc)

def month_name(ym):
    return dt.date(int(ym[:4]), int(ym[4:]), 1).strftime("%b %Y")

# =========================================================================================================
# Title
# =========================================================================================================
footer()
para(f"{BRAND} games · who plays, and from where", 14, True, NAVY, after=0)
para(f"{LM_LONG} · GA4 · built {dt.date.today():%d %b %Y}", 8, False, GREY, after=6)

# =========================================================================================================
# 1. How many games were played each month?
#    One row per month (oldest at the top, report month last and bold) -- identical layout every month.
# =========================================================================================================
question("1. How many games were played each month?")
n_plays = note("plays", "Plays: the number of times a game's play button was pressed. Counts the play events listed in "
                        "the notebook settings (web game page, web homepage block, app game list, app home menu).")
n_plat = note("platform", "Web / App: from GA4's platform. Web = the website on desktop and mobile; App = the iOS and "
                          "Android apps. Total = Web + App.")
n_mom = note("mom", "vs previous month: % change against the month before (same column). ▲ green = more, "
                    "▼ red = fewer, new = zero the month before.")

ROW_MONTHS = MONTHS[-MONTHS_BACK:]

def month_rows(frame, cols, strong_col=None, share_total=None, months=None, no_share=(), compact=False,
               flag_months=()):
    """One row per month (oldest first, report month last and bold). For each column: the number
    (+ its share of share_total under it, if given) and the change vs the month before."""
    rows = []
    for m in (months or ROW_MONTHS):
        i = MONTHS.index(m)
        prev_m = MONTHS[i - 1] if i > 0 else None
        is_lm = m == LM
        cells = []
        for c in cols:
            v = frame.at[m, c]
            strong = is_lm or c == strong_col
            n_size = 7.5 if compact else (8.5 if c == strong_col else 8)
            parts = [(f"{v:,.0f}", n_size, strong, NAVY if strong else DARK_GREY, False)]
            if share_total is not None and c not in no_share:
                tot = share_total.get(m, 0)
                parts.append((f"\n{v / tot * 100:.0f}%" if tot else "\n–", 6 if compact else 6.5, False, GREY, False))
            cells.append(parts)
            cells.append(change_parts(v, frame.at[prev_m, c] if prev_m in frame.index else None,
                                      size=7 if compact else 8, compact=compact))
        lab = label(month_name(m), bold=is_lm, size=8.5 if is_lm else 8)
        if m in flag_months:                         # incomplete data that month (see the table's note)
            lab = lab + [(" ⚠", 7.5, True, DOWN, False)]
        rows.append((lab, cells))
    return rows

monthly = (t.groupby("yearMonth")[["plays_web", "plays_app", "plays"]].sum()
            .reindex(MONTHS, fill_value=0))
rows = month_rows(monthly, ["plays_web", "plays_app", "plays"], strong_col="plays")
sub_noted = [label("Plays", n_plays, 7), label("vs previous month", n_mom, 7)]   # *n markers once per table
sub = [label("Plays", size=7), label("vs previous month", size=7)]
grouped_table(
    label("Month", size=8),
    [("web", label("WEB", n_plat, 7.5), sub_noted),
     ("app", label("APP", size=7.5), sub),
     ("total", label("TOTAL", size=7.5), sub)],
    rows)


# =========================================================================================================
# 2. How many players were there each month?
# =========================================================================================================
question("2. How many players were there each month?")
n_players = note("players", "Players: GA4 users who pressed play at least once that month. A GA4 user is a browser "
                            "(cookie) or an app install, not a person. Total counts each user once, so Web + App can "
                            "be slightly more than Total. A bot that doesn't keep cookies counts as a new player on "
                            "every visit.")
pm = (players_mp.assign(s=players_mp["platform"].map(surface))
                .groupby(["yearMonth", "s"])["totalUsers"].sum().unstack(fill_value=0)
                .reindex(index=MONTHS, columns=["Web", "App"], fill_value=0))
pm["Total"] = players_m.set_index("yearMonth")["totalUsers"].reindex(MONTHS, fill_value=0)
sub_p = [label("Players", n_players, 7), label("vs previous month", n_mom, 7)]
sub_p_plain = [label("Players", size=7), label("vs previous month", size=7)]
grouped_table(
    label("Month", size=8),
    [("web", label("WEB", n_plat, 7.5), sub_p),
     ("app", label("APP", size=7.5), sub_p_plain),
     ("total", label("TOTAL", size=7.5), sub_p_plain)],
    month_rows(pm, ["Web", "App", "Total"], strong_col="Total"))

# =========================================================================================================
# 3. Where did they play? (desktop, mobile, app)
# =========================================================================================================
question("3. Where did they play?")
n_dev = note("device", "Desktop / Mobile: plays on the website, split by GA4's device category (Mobile includes "
                       "tablets). App = the iOS and Android apps, on any device. Under each number: its share of all "
                       "plays that month.")
d = dev.copy()
d["where"] = np.where(d["platform"].str.lower() != "web", "App",
                      np.where(d["deviceCategory"].str.lower() == "desktop", "Desktop", "Mobile"))
d3 = (d.groupby(["yearMonth", "where"])["eventCount"].sum().unstack(fill_value=0)
       .reindex(index=MONTHS, columns=["Desktop", "Mobile", "App"], fill_value=0))
d3_tot = d3.sum(axis=1).to_dict()
sub_d = [label("Plays", n_dev, 7), label("vs previous month", size=7)]
grouped_table(
    label("Month", size=8),
    [("desktop", label("DESKTOP", size=7.5), sub_d),
     ("mobile", label("MOBILE", size=7.5), sub),
     ("app", label("APP", size=7.5), sub)],
    month_rows(d3, ["Desktop", "Mobile", "App"], share_total=d3_tot))

# ---------------------------------------------------------------------------------------------------------
# Split month tables (Q4-Q6): same layout as Q1-Q3 (months = rows), at most GROUPS_PER_TABLE column groups
# per table, so the font stays the same size. Tables of one question follow each other with a small gap.
# ---------------------------------------------------------------------------------------------------------
GROUPS_PER_TABLE = 3
PAL["item"] = PAL["total"]
PAL["muted"] = ("8A9096", "E8EAED", "5F6368", "F8F9FA")

def split_month_tables(frame, cols, header_of, first_sub, months=None, share_total=None, no_share=(),
                       palette_of=lambda c: "item", per_table=GROUPS_PER_TABLE, compact=False, flag_months=()):
    """frame: months × cols. One group per column (number + vs previous month), per_table groups per table.
    compact = smaller numbers and tighter cells, for 6 groups per table (Q4)."""
    chunks = [cols[i:i + per_table] for i in range(0, len(cols), per_table)]
    sub_size = 6.5 if compact else 7
    for k, chunk in enumerate(chunks):
        groups = []
        for j, c in enumerate(chunk):
            first = k == 0 and j == 0
            subs = first_sub if first else [label(t_[0][0], size=sub_size) for t_ in first_sub]
            groups.append((palette_of(c), header_of(c, first), subs))
        rows = month_rows(frame, chunk, share_total=share_total, months=months, no_share=no_share, compact=compact,
                          flag_months=flag_months)
        if compact:
            grouped_table(label("Month", size=8), groups, rows, first_w=1.9, head_size=7.5, sub_size=sub_size,
                          cell_mar=0.05)
        else:
            grouped_table(label("Month", size=8), groups, rows)
        if k < len(chunks) - 1:
            para(after=6)

# =========================================================================================================
# 4. Where are they? (top countries) -- one column group per country, months as rows
# =========================================================================================================
question("4. Where are they?")
Q4_TOP = 5                                            # countries ranked per month; the rest is "Other"
cpiv = (t.pivot_table(index="yearMonth", columns="country", values="plays", aggfunc="sum", fill_value=0)
         .reindex(index=MONTHS, fill_value=0))
PM = MONTHS[-2]

# Each month's own top 5, in rank order
q4_months = ROW_MONTHS
top_of = {}
for m in MONTHS:
    row = cpiv.loc[m]
    row = row[row > 0].sort_values(ascending=False)
    top_of[m] = list(row.index[:Q4_TOP])
rank_of = {m: {c: i + 1 for i, c in enumerate(cpiv.loc[m].sort_values(ascending=False).index)} for m in MONTHS}

# One colour per country, the same in every month (text colour, light fill)
COUNTRY_COLS = [("3949AB", "E8EAF6"), ("00897B", "E0F2F1"), ("8E24AA", "F3E5F5"), ("EF6C00", "FFF3E0"),
                ("D81B60", "FCE4EC"), ("6D4C41", "EFEBE9"), ("1E88E5", "E3F2FD"), ("7CB342", "F1F8E9"),
                ("F4511E", "FBE9E7"), ("546E7A", "ECEFF1"), ("C0CA33", "F9FBE7"), ("5E35B1", "EDE7F6")]
shown = []
for m in q4_months:
    for c in top_of[m]:
        if c not in shown:
            shown.append(c)
shown.sort(key=lambda c: -cpiv[c].loc[q4_months].sum())
col_of = {c: COUNTRY_COLS[i % len(COUNTRY_COLS)] for i, c in enumerate(shown)}

def marker(c, m):
    """★ = first time in the top 5 in this table · ↩ = back after being out the month before."""
    i = q4_months.index(m)
    if i == 0:
        return ""
    prev_m = q4_months[i - 1]
    if c in top_of[prev_m]:
        return ""
    earlier = any(c in top_of[x] for x in q4_months[:i - 1])
    return " ↩" if earlier else " ★"

rows = []
for m in q4_months:
    i = MONTHS.index(m)
    prev_m = MONTHS[i - 1] if i > 0 else None
    is_lm = m == LM
    tot = cpiv.loc[m].sum()
    cells = []
    for r in range(Q4_TOP):
        if r < len(top_of[m]):
            c = top_of[m][r]
            v = cpiv.at[m, c]
            txt, fill = col_of[c]
            cells.append({"parts": [(c, 7, True, txt, False),                      # full country name
                                    (marker(c, m), 7, True, txt, False),
                                    (f"\n{v:,.0f}", 7.5, is_lm, NAVY if is_lm else DARK_GREY, False),
                                    (f" · {v / tot * 100:.0f}%" if tot else " · –", 6, False, GREY, False)],
                          "fill": fill})
            cells.append({"parts": change_parts(v, cpiv.at[prev_m, c] if prev_m else None, size=7, compact=True),
                          "fill": fill})
        else:
            cells += [[("–", 7, False, GREY, False)], [("", 7, False, GREY, False)]]
    oth = tot - cpiv.loc[m, top_of[m]].sum()
    oth_prev = (cpiv.loc[prev_m].sum() - cpiv.loc[prev_m, top_of[prev_m]].sum()) if prev_m else None
    cells.append([(f"{oth:,.0f}", 7.5, is_lm, NAVY if is_lm else DARK_GREY, False),
                  (f"\n{oth / tot * 100:.0f}%" if tot else "\n–", 6, False, GREY, False)])
    cells.append(change_parts(oth, oth_prev, size=7, compact=True))
    left = [c for c in top_of[prev_m] if c not in top_of[m]] if (prev_m and m != q4_months[0]) else []
    left_parts = []
    for k, c in enumerate(left):
        left_parts += [(("\n" if k else "") + c, 6.5, True, col_of.get(c, (DARK_GREY,))[0], False),
                       (f" → #{rank_of[m].get(c, '–')}" if cpiv.at[m, c] > 0 else " → no plays", 6.5, False, GREY, False)]
    cells.append(left_parts or [("", 7, False, GREY, False)])
    rows.append((label(month_name(m), bold=is_lm, size=8 if is_lm else 7.5), cells))

n_cty = note("country", f"Countries: GA4's country, from the visitor's IP address (VPNs and data centres count where "
                        f"the server is). Each row = that month's own top {Q4_TOP} by plays, in rank order, so the "
                        f"countries change from row to row. Each country keeps the same colour in every month. "
                        f"★ = in the top {Q4_TOP} for the first time in this table; ↩ = back in the top {Q4_TOP} after "
                        f"being out the month before. Next to each number: its share of all plays that month. vs prev. = "
                        f"that country's plays vs its own plays the month before (×N = N times as many). Other = every "
                        f"country outside that month's top {Q4_TOP}. Left top {Q4_TOP} = countries in the top {Q4_TOP} "
                        f"the month before but not this month, → their rank now.")

sub_r = [label("Country · plays", size=6.5), label("vs prev.", size=6.5)]
groups = [("total", label(f"#{r + 1}", n_cty if r == 0 else None, 7.5), sub_r) for r in range(Q4_TOP)]
groups.append(("muted", label("OTHER", size=7.5), [label("Plays", size=6.5), label("vs prev.", size=6.5)]))
groups.append(("muted", label(f"LEFT TOP {Q4_TOP}", size=7.5), [label("now", size=6.5)]))
first_w = 1.6
left_w = 1.75
rank_w = (PAGE_W - first_w - left_w - (1.2 + 0.85)) / Q4_TOP     # ≈ 2.5 cm per rank
sub_widths = [rank_w - 0.85, 0.85] * Q4_TOP + [1.2, 0.85, left_w]
grouped_table(label("Month", size=8), groups, rows, first_w=first_w, head_size=7.5, sub_size=6.5,
              cell_mar=0.05, sub_widths=sub_widths)

# =========================================================================================================
# 5. How did they land here? (traffic sources)
# =========================================================================================================
question("5. How did they land here?")
# Same source lists as the weekly brief (OLJ)
SRC_GROUPS = ["Direct", "Search Engines", "Social Networks", "AI Assistants", "Internal/Newsletters", "Other"]
SRC_LISTS = {
    "Direct": {"(direct)"},
    "Search Engines": {"google", "news.google.com", "bing", "ecosia.org", "qwant.com", "duckduckgo",
                       "fr.search.yahoo.com", "yahoo", "search.brave.com", "yandex", "ya.ru", "startpage.com"},
    "Social Networks": {"m.facebook.com", "facebook.com", "l.facebook.com", "lm.facebook.com", "mobile.facebook.com",
                        "facebook", "l.instagram.com", "ig", "instagram", "instagram.com", "later-linkinbio",
                        "linkin.bio", "t.co", "linkedin.com", "lnkd.in", "go.bsky.app", "l.threads.com", "twitter",
                        "x.com", "threads", "bluesky", "pinterest.com", "tiktok.com", "snapchat", "snapchat.com",
                        "web.whatsapp.com", "cms-46", "reddit.com", "fb", "flipboard", "flipboard.com"},
    "AI Assistants": {"chatgpt.com", "perplexity.ai", "gemini.google.com", "perplexity", "copilot.com",
                      "copilot.microsoft.com", "openai", "duck.ai", "chat.mistral.ai", "notebooklm.google.com",
                      "claude.ai", "poe.com", "grok.com", "chat.qwen.ai", "doubao.com", "chat.z.ai", "felo.ai",
                      "mammouth.ai", "you.com"},
    "Internal/Newsletters": {"mailchimp", "newsletter", "email", "website", "olj", "google-play", "olj.me", "olj.wael",
                             "autopromoolj", "morningbrief", "hs_email", "brevo", "mailchi.mp",
                             "us1.campaign-archive.com", "actito.be", "activetrail", "acumbamail", "omnisend",
                             "wordfly", "newsletter_1", "newsletter_6b", "newsletter_paiementechouepp",
                             "newsletter_preventif", "partenairesjamhour", "marketo",
                             "gmi mailchimp integration prod list", "master list", "bundle", "nb",
                             "cms-9"},   # cms-9 = OLJ newsletter
}
def src_group(x):
    x = str(x).strip().lower()
    for g, names in SRC_LISTS.items():
        if x in names:
            return g
    return "Other"

sc = src.assign(group=src["sessionSource"].map(src_group))
sframe = (sc.pivot_table(index="yearMonth", columns="group", values="eventCount", aggfunc="sum", fill_value=0)
            .reindex(index=MONTHS, columns=SRC_GROUPS, fill_value=0))
other_top = (sc[(sc["group"] == "Other") & (sc["yearMonth"] == LM)].groupby("sessionSource")["eventCount"].sum()
               .sort_values(ascending=False).head(5))
other_txt = ", ".join(f"{s_} ({v:,.0f})" for s_, v in other_top.items()) or "none"
n_src = note("sources", f"Sources: each play gets the source of the visit it happened in (GA4 session source), "
                        f"grouped with the same lists as the weekly brief (see Source groups below; NEWSLETTERS = Internal/Newsletters). ×N = N times as many plays as the month before. Under each "
                        f"number: its share of all plays that month. Biggest Other sources in {month_name(LM)}: "
                        f"{other_txt}.")
# One table, one colour per source group (the weekly brief's category colours)
SRC_PALS = {   # group header · sub-header fill · sub-header text · cell fill
    "Direct":               ("4285F4", "D9E6FC", "1A5BB8", "F0F5FE"),
    "Search Engines":       ("00A891", "CCF2EC", "00796B", "EEFAF8"),
    "Social Networks":      ("9C27B0", "EBD4F0", "6A1B9A", "F8F0FA"),
    "AI Assistants":        ("F59300", "FFE5C2", "B35F00", "FFF6EB"),
    "Internal/Newsletters": ("E8538A", "FFD9E6", "C2185B", "FFF0F5"),
    "Other":                PAL["muted"],
}
SRC_SHORT = {"Direct": "DIRECT", "Search Engines": "SEARCH", "Social Networks": "SOCIAL",
             "AI Assistants": "AI ASSISTANTS", "Internal/Newsletters": "NEWSLETTERS", "Other": "OTHER"}
split_month_tables(sframe, SRC_GROUPS, lambda g, first: label(SRC_SHORT[g], n_src if first else None, 7.5),
                   [label("Plays", size=6.5), label("vs prev.", size=6.5)],
                   share_total=sframe.sum(axis=1).to_dict(),
                   palette_of=lambda g: SRC_PALS[g], per_table=6, compact=True)

# =========================================================================================================
# 6. How many accounts were created because of games? (CMS)
# =========================================================================================================
question("6. How many accounts were created because of games?")
n_acc = note("game_accounts", "Accounts: accounts created in the CMS whose acquisition source contains a game's name "
                              "(motamo, sudoku, prefixo, synonymo, decrypto, melimelo; case and accents ignored), by "
                              "creation date in Beirut time. Under each number: its share of all new accounts that month."
                              + ("" if not PARTIAL else
                                 " ⚠ = incomplete month: on " + "; ".join(
                                     f"{a:%d %b %Y}–{b:%d %b}" + f" the CMS returned only {g:,} of {n:,} accounts"
                                     for a, b, g, n in PARTIAL)
                                 + " (its ~1,000-row limit per request), so that month is undercounted."))
n_acc_all = note("all_accounts", "All accounts: every new OLJ account created that month, whatever its acquisition "
                                 "source. OT accounts (English preferred language, as in the weekly brief) are left out; "
                                 "game accounts always count, since the games are OLJ-only.")
aframe = GAME_ACC.T.reindex(columns=GAMES).copy()
aframe.columns = [g.capitalize() for g in GAMES]
aframe["All games"] = GAME_ACC.sum().values
aframe["All new accounts"] = ALL_ACC.values
# One table, one colour per game; All games in navy, all new accounts in grey
GAME_PALS = {   # group header · sub-header fill · sub-header text · cell fill
    "Motamo":   ("3949AB", "DADDF3", "283593", "F2F3FB"),    # indigo
    "Sudoku":   ("00897B", "CDEBE7", "00695C", "EEF8F6"),    # teal
    "Prefixo":  ("8E24AA", "EBD3F1", "6A1B9A", "F8F0FA"),    # purple
    "Synonymo": ("EF6C00", "FDE0C8", "B35F00", "FFF6EB"),    # orange
    "Decrypto": ("D81B60", "F8D3E0", "AD1457", "FDF0F5"),    # pink
    "Melimelo": ("6D4C41", "E6DCD8", "4E342E", "F7F3F1"),    # brown
    "All games": PAL["total"],
    "All new accounts": PAL["muted"],
}
def acc_header(c, first):
    if c == "All new accounts":
        return label("ALL ACCOUNTS", n_acc_all, 7.5)
    return label(c.upper(), n_acc if first else None, 7.5)
# 8 groups (6 games + all games + all accounts) in one compact table
split_month_tables(aframe, list(aframe.columns), acc_header,
                   [label("Accounts", size=6.5), label("vs prev.", size=6.5)],
                   months=ACC_MONTHS, share_total=ALL_ACC.to_dict(), no_share=("All new accounts",),
                   palette_of=lambda c: GAME_PALS.get(c, "item"), per_table=8, compact=True,
                   flag_months=PARTIAL_MONTHS)

# =========================================================================================================
# Appendix: notes
# =========================================================================================================
para("Notes", 11, True, NAVY, before=16, after=4, keep_next=True)
for _, (n, text) in sorted(NOTES.items(), key=lambda kv: kv[1][0]):
    p = para(after=2)
    run(p, f"*{n}  ", 8, True, NAVY)
    run(p, text, 8, False, DARK_GREY)

para("Source groups", 11, True, NAVY, before=12, after=4, keep_next=True)
grouped_table(label("Group", size=8),
              [("total", label("SOURCE NAMES COUNTED", size=7.5), [label("GA4 session source", size=7)])],
              [(label(g, size=8), [[(", ".join(sorted(SRC_LISTS[g])), 7, False, DARK_GREY, False)]])
               for g in SRC_LISTS] +
              [(label("Other", size=8, color=GREY), [[("anything not listed above", 7, False, GREY, False)]])],
              first_w=3.8)

REPORT_DOCX = f"{BRAND}_Games_by_Country_{LM[:4]}-{LM[4:]}.docx"
doc.save(REPORT_DOCX)
print("Saved:", REPORT_DOCX)
try:
    from google.colab import files
    files.download(REPORT_DOCX)
except Exception:
    pass

## Assumptions & caveats

- **Events** are the ones listed in ⚙️ Settings, assumed shipped and named exactly so in GA4. Step 3 shows any that never appear.
- **Web vs App** comes from GA4's `platform` (web stream → Web, iOS/Android streams → App), not from the event name. A game page
  opened in the app's webview that fires the website's tag counts as Web.
- **Players** (`totalUsers` filtered to play events) are de-duplicated within a month × country. The Web and App player columns
  can add up to slightly more than the total, because one person can use both.
- **Country** is GA4's IP-based country. VPNs and data-centre bots often show up in the US, Netherlands, Germany, Ireland, or Singapore, so watch those.
- GA4 may apply **data thresholds** (when Google signals is on) and hide small rows. Small countries can show gaps.
- Plays from the homepage block and from the game page are both counted. If the homepage button only *navigates* to the game page,
  that double counts. Delete the `_hp` lines in ⚙️ Settings once you've checked.